In [5]:
import pandas as pd
import numpy as np

df_A = pd.read_csv('data/attacker_turns.csv')
df_D = pd.read_csv('data/defender_turns.csv')

df = pd.merge(df_A, df_D, how='outer', on=[
              'game_id', 'class_alias', 'attacker', 'defender', 'turn'], suffixes=('_A', '_D'))

In [6]:
SA_a_success = 3
SA_a_compiles = 5

SA_d_success = 3
SA_d_compiles = 5

turn_limit = 20

In [7]:
metrics = []
for (class_alias, attacker, defender), data in df.groupby(['class_alias', 'attacker', 'defender']):
    A_success = data[data['success_A'].notna()]
    A_compilable = data[data['compilable_A'] != 0]

    mut_live_attempts = 0

    if A_success.shape[0] > 0:
        mut_live_attempts = A_success['compilable_A'].sum() / A_success.shape[0]
    else:
        mut_live_attempts = np.nan

    mut_compiles_attempts = 0
    if A_compilable.shape[0] > 0:
        for ind, row in A_success.iterrows():
            mut_compiles_attempts += (row['attempts_A'] - (row['success_A'] - row['compilable_A']) * SA_a_compiles) / row['compilable_A']

        mut_compiles_attempts_right = 0
        for ind, row in A_compilable[A_compilable['success_A'].isna()].iterrows():
            mut_compiles_attempts_right += (row['attempts_A'] - (SA_a_success - row['compilable_A']) * SA_a_compiles) / row['compilable_A']

        mut_compiles_attempts += mut_compiles_attempts_right
        mut_compiles_attempts /= A_compilable.shape[0]
    else:
        mut_compiles_attempts = np.nan

    mut_compiles_failure = A_success.apply(lambda x: x['success_A'] - x['compilable_A'], axis=1).sum()
    for ind, row in data[data['success_A'].isna()].iterrows():
        mut_compiles_failure += (SA_a_success - row['compilable_A'])
    mut_compiles_failure /= turn_limit

    D_success = data[data['success_D'].notna()]
    D_compilable = data[(data['compilable_D'].notna())
                        & (data['compilable_D'] != 0)]
    D_valid = data[data['attempts_D'].notna()]

    test_killing_attempts = 0
    if D_success.shape[0] > 0:
        test_killing_attempts = D_success['compilable_D'].sum() / D_success.shape[0]
    else:
        test_killing_attempts = np.nan

    test_compiles_attempts = 0
    if D_compilable.shape[0] > 0:
        for ind, row in D_success.iterrows():
            test_compiles_attempts += (row['attempts_D'] - (row['success_D'] - row['compilable_D']) * SA_d_compiles) / row['compilable_D']

        test_compiles_attempts_right = 0
        for ind, row in D_compilable[D_compilable['success_D'].isna()].iterrows():
            test_compiles_attempts_right += (row['attempts_D'] - (SA_d_success - row['compilable_D']) * SA_d_compiles) / row['compilable_D']

        test_compiles_attempts += test_compiles_attempts_right
        test_compiles_attempts /= D_compilable.shape[0]
    else:
        test_compiles_attempts = np.nan

    if D_valid.shape[0] > 0:
        test_compiles_failure = D_success.apply(lambda x: x['success_D'] - x['compilable_D'], axis=1).sum()
        for ind, row in D_valid[D_valid['success_D'].isna()].iterrows():
            test_compiles_failure += (SA_d_success - row['compilable_D'])
        test_compiles_failure /= D_valid.shape[0]
    else:
        test_compiles_failure = np.nan


    metrics.append({
        'class_alias': class_alias,
        'attacker': attacker,
        'defender': defender,
        'mut_live_attempts': mut_live_attempts,
        'mut_compiles_attempts': mut_compiles_attempts,
        'mut_compiles_failure': mut_compiles_failure,
        'test_killing_attempts': test_killing_attempts,
        'test_compiles_attempts': test_compiles_attempts,
        'test_compiles_failure': test_compiles_failure
    })

df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv('data/diagnostic_metrics.csv', index=False)